# Do you know when you know? — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/03-function-sampling.ipynb)

Reproduces the fixed 20-round function-sampling benchmark. It loads the exact formulas, reveal order, plot images, prompts and saved model decisions used in the article. No paid request runs unless `RUN_MODEL` is set explicitly.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    candidates = [Path("../") / relative_path, Path(relative_path)]
    for path in candidates:
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/03-function-sampling/benchmark.json")
results = load_json("data/03-function-sampling/results.json")
print(f"{len(benchmark['rounds'])} rounds; {len(results['models'])} completed model runs")


## Inspect the exact sampling sequence

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

VARIANT_TEXT = {
    "linear-1": "f(x) = x", "linear-3": "f(x) = 3x",
    "square-1": "f(x) = x²", "square-4": "f(x) = 4x²",
    "log-1": "f(x) = ln(1 + x)", "log-4": "f(x) = ln(1 + 4x)",
    "reciprocal-1": "f(x) = 1 / x", "reciprocal-3": "f(x) = 3 / x",
    "hump-1": "f(x) = 4x(1 − x)", "hump-2": "f(x) = 8x(1 − x)",
    "valley-1": "f(x) = |2x − 1|", "valley-3": "f(x) = 3|2x − 1|",
    "sine-2pi": "f(x) = sin(2πx)", "sine-4pi": "f(x) = 2sin(4πx)",
    "plateau-1": "f(x) = min(2x, 1)", "plateau-3": "f(x) = 3 min(2x, 1)",
}

def read_bytes(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists(): return path.read_bytes()
    with urllib.request.urlopen(RAW + relative_path) as response: return response.read()

def state_image(round_number, samples):
    path = f"data/03-function-sampling/stimuli/round-{round_number:02d}/samples-{samples:02d}.png"
    return Image.open(BytesIO(read_bytes(path))).convert("RGB")

round_number = 1
round_data = benchmark["rounds"][round_number - 1]
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for samples, ax in enumerate(axes, start=1):
    ax.imshow(state_image(round_number, samples)); ax.axis("off"); ax.set_title(f"{samples} sample(s)")
plt.show()
for card in round_data["cards"]:
    print(f'{card["label"]}: {VARIANT_TEXT[card["variantId"]]}')
print("Expected:", "NOT DISTINGUISHABLE" if len(round_data["hiddenLabels"]) > 1 else "OPTION " + round_data["hiddenLabels"][0])


## Exact model prompt

In [ ]:
SYSTEM_PROMPT = """You are playing a visual function-identification game. A plot shows the currently revealed observations; four candidate formulas are listed in the accompanying text. Axes have no fixed units: identify qualitative shape, not coefficient scale. Return exactly one command and nothing else: OPTION A, OPTION B, OPTION C, OPTION D, SAMPLE MORE, or NOT DISTINGUISHABLE.

Use SAMPLE MORE when current evidence is insufficient but another observation could separate the options. Use NOT DISTINGUISHABLE only when multiple options represent the same qualitative shape under independent positive axis scaling, so no future observation can uniquely separate them. At most six samples are available."""

def state_text(round_data, samples):
    cards = "\n".join(f'{c["label"]}: {VARIANT_TEXT[c["variantId"]]}' for c in round_data["cards"])
    return f"Visible samples: {samples} of 6.\n{cards}\nReturn exactly one valid command."

print(SYSTEM_PROMPT)
print("\n--- example user message ---\n")
print(state_text(round_data, 3))


## Recompute the published table

In [ ]:
import pandas as pd

rows = []
for run in results["models"]:
    rounds = run["rounds"]
    correct = [r for r in rounds if r["correct"]]
    false_nonunique = sum(r["finalCommand"] == "NOT DISTINGUISHABLE" and r["expectedCommand"] != "NOT DISTINGUISHABLE" for r in rounds)
    rows.append({
        "model": run["model"],
        "correct": f'{len(correct)}/20',
        "mean samples on correct": round(sum(r["samplesUsed"] for r in correct) / len(correct), 2),
        "false non-unique calls": false_nonunique,
        "cost (USD)": round(sum(r["costUsd"] for r in rounds), 4),
    })
pd.DataFrame(rows).sort_values(["correct", "mean samples on correct"], ascending=[False, True]).reset_index(drop=True)


## Full opt-in API replication

The cells below run the exact published tasks across all published models (20 adaptive rounds × 6 models, up to 720 fresh state calls), save every request and raw response, record provider, tokens, latency and cost, and resume from completed calls.

Paid calls are off by default. Add `OPENROUTER_API_KEY` to Colab Secrets, set `RUN_PAID_EVAL = True`, and choose an aggregate budget. The historical run cost about $1.36; current prices and routing can differ. Each request reserves a conservative worst-case amount before it is sent, so the configured aggregate budget cannot be exceeded.


In [ ]:
import base64, hashlib, re

BENCHMARK_ID = benchmark["benchmarkId"]
RUN_PAID_EVAL = False
BUDGET_USD = 5.0
RERUN_DIR = Path("function-sampling-rerun")
MODEL_SPECS = {
    "openai/gpt-5.6-sol": {"input": 2.0, "output": 10.0},
    "google/gemini-3.7-flash": {"input": 0.75, "output": 3.75},
    "anthropic/claude-opus-5": {"input": 5.0, "output": 25.0},
    "anthropic/claude-sonnet-5": {"input": 2.0, "output": 10.0},
    "qwen/qwen3.8-max": {"input": 2.0, "output": 6.0},
    "moonshotai/kimi-k3": {"input": 3.0, "output": 15.0},
    "z-ai/glm-5.3-flash": {"input": 0.075, "output": 0.25},
}
MODELS_TO_RUN = [run["model"] for run in results["models"]]
RUN_SETTINGS = {"max_tokens": 8192, "reasoning": {"effort": "low"}, "temperature": 0}
VALID_COMMANDS = {"OPTION A", "OPTION B", "OPTION C", "OPTION D", "SAMPLE MORE", "NOT DISTINGUISHABLE"}

def expected_command(round_data):
    return "NOT DISTINGUISHABLE" if len(round_data["hiddenLabels"]) > 1 else "OPTION " + round_data["hiddenLabels"][0]

def parse_command(text):
    normalized = (text or "").strip().upper()
    return (normalized if normalized in VALID_COMMANDS else None), normalized in VALID_COMMANDS

def state_png(round_number, samples):
    return read_bytes(f"data/03-function-sampling/stimuli/round-{round_number:02d}/samples-{samples:02d}.png")

def build_state_payload(model, round_data, samples):
    image = base64.b64encode(state_png(round_data["round"], samples)).decode()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "text", "text": state_text(round_data, samples)},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + image}},
        ]},
    ]
    return openrouter_payload(model, messages, **RUN_SETTINGS)


In [ ]:
import hashlib, os, time, urllib.error
from datetime import datetime, timezone

OPENROUTER_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
MAX_NOTEBOOK_BUDGET_USD = 25.0

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(encoded.encode()).hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")

def append_jsonl(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as handle:
        handle.write(json.dumps(value, ensure_ascii=False) + "\n")

def get_api_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("OpenRouter API key: ")
    return key

def openrouter_payload(model, messages, *, max_tokens, reasoning, temperature=None):
    spec = MODEL_SPECS[model]
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "reasoning": reasoning,
        "usage": {"include": True},
        "provider": {
            "sort": "price", "allow_fallbacks": False, "require_parameters": False,
            "max_price": {"prompt": spec["input"], "completion": spec["output"]},
        },
    }
    if temperature is not None:
        payload["temperature"] = temperature
    return payload

def reserve_cost(payload):
    spec = MODEL_SPECS[payload["model"]]
    # UTF-8 bytes safely overestimate prompt tokens, including encoded images.
    prompt_ceiling = len(json.dumps(payload["messages"], ensure_ascii=False).encode()) + 256
    return prompt_ceiling * spec["input"] / 1_000_000 + payload["max_tokens"] * spec["output"] / 1_000_000

def send_openrouter(payload, api_key, timeout=240):
    request = urllib.request.Request(
        OPENROUTER_ENDPOINT,
        data=json.dumps(payload).encode(),
        method="POST",
        headers={
            "Authorization": f"Bearer {api_key}", "Content-Type": "application/json",
            "HTTP-Referer": "https://alestainer.com", "X-OpenRouter-Title": BENCHMARK_ID,
        },
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status, body = response.status, response.read().decode()
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read().decode(errors="replace")
    try:
        parsed = json.loads(body)
    except json.JSONDecodeError:
        parsed = {"unparsed_body": body}
    return status, parsed, time.perf_counter() - started

def content_of(response):
    try:
        content = response["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return None
    return content if isinstance(content, str) else None

def usage_of(response):
    usage = response.get("usage") if isinstance(response.get("usage"), dict) else {}
    details = usage.get("completion_tokens_details")
    details = details if isinstance(details, dict) else {}
    cost = usage.get("cost")
    reasoning = details.get("reasoning_tokens")
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": reasoning if isinstance(reasoning, (int, float)) else None,
        "cost_usd": cost if isinstance(cost, (int, float)) else None,
        "raw_usage": usage,
    }

def result_files():
    return sorted(RERUN_DIR.glob("models/*/calls/*/result.json"))

def load_existing():
    existing = {}
    for path in result_files():
        row = json.loads(path.read_text())
        if row.get("suite_hash") != SUITE_HASH:
            raise RuntimeError(f"Suite mismatch in {path}")
        existing[(row["model"], row["call_id"])] = row
    return existing

def prepare_run():
    if BUDGET_USD <= 0 or BUDGET_USD > MAX_NOTEBOOK_BUDGET_USD:
        raise ValueError(f"BUDGET_USD must be above $0 and at most ${MAX_NOTEBOOK_BUDGET_USD}")
    RERUN_DIR.mkdir(parents=True, exist_ok=True)
    config_path = RERUN_DIR / "config.json"
    config = {
        "created_at": utc_now(), "benchmark_id": BENCHMARK_ID,
        "models": MODELS_TO_RUN, "model_specs": MODEL_SPECS,
        "suite_hash": SUITE_HASH, "run_settings": RUN_SETTINGS,
        "approved_aggregate_budget_usd": BUDGET_USD,
    }
    if config_path.exists():
        prior = json.loads(config_path.read_text())
        for key in ("benchmark_id", "models", "model_specs", "suite_hash", "run_settings"):
            if prior.get(key) != config.get(key):
                raise RuntimeError(f"Resume configuration mismatch: {key}")
        if BUDGET_USD > float(prior["approved_aggregate_budget_usd"]):
            raise RuntimeError("A resume cannot raise the original budget; use a new output directory")
    else:
        write_json(config_path, config)
    existing = load_existing()
    if any(row.get("cost_usd") is None for row in existing.values()):
        raise RuntimeError("Cannot resume safely: a completed call is missing reported cost")
    return get_api_key(), existing, sum(float(row["cost_usd"]) for row in existing.values())

def save_call(*, model, call_id, payload, expected, metadata, existing, api_key, spent):
    key = (model, call_id)
    if key in existing:
        return existing[key], spent, None
    reservation = reserve_cost(payload)
    if spent + reservation > BUDGET_USD + 1e-12:
        return None, spent, f"STOP budget: spent ${spent:.6f}; next call reserves ${reservation:.6f}; cap ${BUDGET_USD:.6f}"
    call_dir = RERUN_DIR / "models" / model.replace("/", "--") / "calls" / call_id
    write_json(call_dir / "request.json", payload)
    status, response, latency = send_openrouter(payload, api_key)
    write_json(call_dir / "raw-response.json", response)
    append_jsonl(RERUN_DIR / "raw-responses.jsonl", {
        "model": model, "call_id": call_id, "http_status": status, "response": response,
    })
    if status < 200 or status >= 300 or "error" in response:
        write_json(call_dir / "error.json", {
            "model": model, "call_id": call_id, "http_status": status,
            "latency_seconds": latency, "response": response, "recorded_at": utc_now(),
        })
        return None, spent, f"STOP API error on {model}/{call_id}: HTTP {status}; rerun to resume"
    raw = content_of(response)
    parsed, strict = parse_command(raw)
    usage = usage_of(response)
    result = {
        "model": model, "returned_model": response.get("model"),
        "provider": response.get("provider"), "call_id": call_id,
        "suite_hash": SUITE_HASH, "expected": expected, "parsed": parsed,
        "strictly_formatted": strict, "correct": parsed == expected,
        "raw_content": raw, "latency_seconds": latency, "recorded_at": utc_now(),
        **metadata, **usage,
    }
    write_json(call_dir / "result.json", result)
    existing[key] = result
    if usage["cost_usd"] is None:
        return result, spent, "STOP accounting: response has no reported cost; rerun to resume"
    spent += float(usage["cost_usd"])
    return result, spent, None


In [ ]:
SUITE_HASH = stable_hash({
    "benchmark": BENCHMARK_ID,
    "rounds": [(r["round"], expected_command(r), [hashlib.sha256(state_png(r["round"], s)).hexdigest() for s in range(1, 7)]) for r in benchmark["rounds"]],
    "models": MODELS_TO_RUN, "settings": RUN_SETTINGS,
})

def run_full_sequential_suite():
    api_key, existing, spent = prepare_run()
    stop = None
    for model_index, model in enumerate(MODELS_TO_RUN):
        for round_data in benchmark["rounds"]:
            expected = expected_command(round_data)
            samples = 1
            while True:
                call_id = f"round-{round_data['round']:02d}--samples-{samples:02d}"
                if (model, call_id) not in existing:
                    print(f"run {model} / {call_id}")
                result, spent, stop = save_call(
                    model=model, call_id=call_id,
                    payload=build_state_payload(model, round_data, samples), expected=expected,
                    metadata={"round": round_data["round"], "samples_visible": samples},
                    existing=existing, api_key=api_key, spent=spent,
                )
                if stop:
                    print(stop)
                    break
                command = result.get("parsed")
                if command == "SAMPLE MORE" and samples < benchmark["maxSamples"]:
                    samples += 1
                    continue
                round_rows = [row for (m, _), row in existing.items() if m == model and row.get("round") == round_data["round"]]
                summary = {
                    "model": model, "round": round_data["round"], "final_command": command,
                    "expected_command": expected, "correct": command == expected,
                    "samples_used": samples,
                    "cost_usd": sum(float(row["cost_usd"]) for row in round_rows if row.get("cost_usd") is not None),
                }
                out = RERUN_DIR / "models" / model.replace("/", "--") / "rounds" / f"round-{round_data['round']:02d}.json"
                write_json(out, summary)
                break
            if stop:
                break
        if stop:
            break
    round_files = sorted(RERUN_DIR.glob("models/*/rounds/round-*.json"))
    summary = {
        "updated_at": utc_now(), "completed_state_calls": len(existing),
        "completed_rounds": len(round_files), "expected_rounds": len(MODELS_TO_RUN) * len(benchmark["rounds"]),
        "reported_cost_usd": sum(float(r["cost_usd"]) for r in existing.values() if r.get("cost_usd") is not None),
    }
    write_json(RERUN_DIR / "summary.json", summary)
    return summary

possible_payloads = [build_state_payload(model, round_data, samples) for model in MODELS_TO_RUN for round_data in benchmark["rounds"] for samples in range(1, 7)]
print(f"Exact rounds: {len(benchmark['rounds'])}; selected models: {len(MODELS_TO_RUN)}; at most {len(possible_payloads)} state calls")
print(f"Largest conservative single-call reservation: ${max(map(reserve_cost, possible_payloads)):.4f}")
if RUN_PAID_EVAL:
    print(json.dumps(run_full_sequential_suite(), indent=2))
else:
    print("DRY RUN ONLY — set RUN_PAID_EVAL = True to send requests")


## Analyze your rerun


In [ ]:
round_files = sorted(RERUN_DIR.glob("models/*/rounds/round-*.json"))
if not round_files:
    print("No rerun results yet.")
else:
    rerun = pd.DataFrame(json.loads(path.read_text()) for path in round_files)
    display(rerun.groupby("model").agg(
        rounds=("round", "size"), correct=("correct", "sum"),
        mean_samples=("samples_used", "mean"), cost_usd=("cost_usd", "sum"),
    ))
